# VideoMamba-Tiny — Kaggle CUDA Wheel Builder & Compatibility Validator

**Purpose:** Build reusable CUDA extension wheels for the **original VideoMamba / BiMamba implementation** used by the EgoEvGesture training notebook. This notebook does **not** train a model, read the gesture dataset, or create train/test splits.

It builds **`causal-conv1d 1.0.0`** and **`mamba-ssm 1.0.1`** from the **vendored source in the pinned OpenGVLab/VideoMamba commit** `37355c26d0ae99ca2459f6d4044a5f509031a79f`. This is crucial because generic/upgraded `mamba-ssm` wheels may not implement the `bimamba` argument required by VideoMamba-Tiny.

**Why a new build may be necessary:** compiled PyTorch extensions depend on more than the wheel's Python tag. A change in **Python ABI, PyTorch build, CUDA runtime/toolkit, PyTorch C++ ABI, or GPU architecture** may require rebuilding. A newly compiled wheel is **not guaranteed** to work if the older CUDA sources cannot compile against a newer runtime; failures are reported explicitly.

**Kaggle prerequisites:** GPU enabled; Internet enabled to fetch the pinned GitHub source and dependencies; `nvcc` compiler installed and accessible; sufficient CPU RAM/disk for C++/CUDA builds. You do **not** need to attach the 207 GB EgoEvGesture dataset.

**Outputs:** two `.whl` files, environment/compatibility manifest, SHA-256 checksums, source patch diff, logs, and a separate `verify_install.py` helper for future Kaggle sessions. Place the output directory in a **private Kaggle Dataset** after validation. The training notebook can then Add Input the wheel dataset.

**Status policy:** `BUILD PASS` means two wheel files were successfully created; `RUNTIME PASS` additionally requires a CUDA forward/backward smoke test. Publish or reuse wheels only after runtime validation, and only in an environment that matches the manifest.

## 1. Build configuration

The CUDA architectures are detected from the GPUs in the **current Kaggle runtime**. Override `TARGET_SMS` only if you deliberately build for another target and have a compatible CUDA compiler. `MAX_JOBS=1` is the conservative default to reduce build-time memory pressure.

**Keep `VIDEO_MAMBA_COMMIT` fixed** to match your training notebook. Changing source versions requires rebuilding and revalidating the complete experiment environment.

In [ ]:
from pathlib import Path
import os, sys, re, json, platform, hashlib, subprocess, shutil, time, zipfile, difflib, shlex
from datetime import datetime, timezone

VIDEO_MAMBA_URL = 'https://github.com/OpenGVLab/VideoMamba.git'
VIDEO_MAMBA_COMMIT = '37355c26d0ae99ca2459f6d4044a5f509031a79f'
REPO_DIR = Path('/kaggle/working/VideoMamba_wheel_source')
OUTPUT_DIR = Path('/kaggle/working/videomamba_cuda_wheels')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Leave empty to build only for the detected GPU(s); for example, '75' for Tesla T4.
TARGET_SMS = ''
MAX_JOBS = 1
CHECKOUT_SOURCE_FROM_INPUT = ''    # Optional local path to the exact pinned git repository
INSTALL_BUILT_WHEELS = True       # Install the two wheels into the current runtime for verification
RUN_GPU_SMOKE_TEST = True        # Forward/backward CUDA checks for both packages
INSTALL_LEGACY_IMPORT_DEPS = True  # Install historical transformers/einops dependencies for BiMamba import

# A local version tag makes each binary build identifiable; neither PyTorch nor CUDA is upgraded.
OUTPUT_MANIFEST = OUTPUT_DIR / 'wheel_manifest.json'

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path, block=8*1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(block), b''):
            h.update(chunk)
    return h.hexdigest()

def run_logged(command, log_name, *, cwd=None, env=None, check=True):
    command = [str(x) for x in command]
    log_path = OUTPUT_DIR / log_name
    print('\n$ ' + shlex.join(command), flush=True)
    with log_path.open('w', encoding='utf-8') as log:
        log.write('COMMAND: ' + shlex.join(command) + '\nSTARTED: ' + utc_now() + '\n\n')
        log.flush()
        process = subprocess.Popen(
            command, cwd=str(cwd) if cwd else None, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, errors='replace', bufsize=1,
        )
        for line in process.stdout:
            log.write(line)
            print(line, end='', flush=True)
        rc = process.wait()
        log.write(f'\nRETURN CODE: {rc}\nFINISHED: {utc_now()}\n')
    if check and rc != 0:
        raise RuntimeError(f'Command failed with exit code {rc}; see {log_path}')
    return rc

print('Source commit:', VIDEO_MAMBA_COMMIT)
print('Output folder:', OUTPUT_DIR)

## 2. Runtime audit: Python, PyTorch, CUDA, GPU, and compiler

We inspect **the active runtime**, not a hardcoded Kaggle configuration. Do not confuse `nvidia-smi`'s *driver-supported CUDA version* with the installed `nvcc` toolkit version or `torch.version.cuda`.

A CUDA toolkit is **required to build from source**. Installing only CUDA runtime Python packages is not sufficient. The notebook does **not** downgrade or replace PyTorch.

In [ ]:
import torch
from packaging.version import Version
from torch.utils.cpp_extension import CUDA_HOME

if platform.system() != 'Linux' or platform.machine() not in ('x86_64', 'AMD64'):
    raise RuntimeError('This builder targets Linux x86_64 Kaggle GPU environments.')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU unavailable. Enable a GPU accelerator in Kaggle settings.')
if not torch.version.cuda:
    raise RuntimeError('The current PyTorch was not built with CUDA support.')

nvcc_path = shutil.which('nvcc')
if not nvcc_path and CUDA_HOME:
    candidate = Path(CUDA_HOME) / 'bin' / 'nvcc'
    if candidate.is_file(): nvcc_path = str(candidate)
if not nvcc_path:
    raise RuntimeError('CUDA nvcc compiler not found. This runtime cannot build CUDA wheels from source. Select a Kaggle GPU image with CUDA Toolkit/nvcc installed.')

nvcc_version_txt = subprocess.check_output([nvcc_path, '--version'], text=True, errors='replace')
match = re.search(r'release\s+(\d+\.\d+)', nvcc_version_txt)
if not match:
    raise RuntimeError('Cannot determine CUDA Toolkit version from nvcc --version.')
nvcc_version = match.group(1)
torch_cuda = str(torch.version.cuda)
if Version(nvcc_version).major != Version(torch_cuda).major:
    raise RuntimeError(f'CUDA major version mismatch: nvcc={nvcc_version}, torch CUDA={torch_cuda}. Use a matching CUDA toolkit instead of building incompatible wheels.')
if nvcc_version != torch_cuda:
    print(f'WARNING: Toolkit nvcc {nvcc_version} differs from torch CUDA {torch_cuda}. Minor mismatches may work but are not guaranteed.')

GPU_LIST = []
for i in range(torch.cuda.device_count()):
    cap = torch.cuda.get_device_capability(i)
    GPU_LIST.append({'index': i, 'name': torch.cuda.get_device_name(i), 'sm': f'{cap[0]}{cap[1]}'})
requested_sms = sorted({x.strip().replace('.', '').replace('sm_', '') for x in TARGET_SMS.split(',') if x.strip()} or {x['sm'] for x in GPU_LIST})
if not requested_sms or any(not re.fullmatch(r'\d{2,3}', x) for x in requested_sms):
    raise ValueError(f'Invalid TARGET_SMS: {TARGET_SMS!r}')

# Explicit NVCC support check. CUDA toolkits can drop architectures over time.
arch_text = subprocess.run([nvcc_path, '--list-gpu-code'], capture_output=True, text=True, check=True).stdout
unsupported = [sm for sm in requested_sms if f'sm_{sm}' not in arch_text]
if unsupported:
    raise RuntimeError(f'nvcc {nvcc_version} does not advertise support for GPU targets {unsupported}. Supported codes: {arch_text[:900]}')

abi = int(bool(torch._C._GLIBCXX_USE_CXX11_ABI))
python_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
torch_full = torch.__version__
torch_base = torch_full.split('+')[0]
local_tag = ('vm' + VIDEO_MAMBA_COMMIT[:8] + '.cu' + torch_cuda.replace('.', '') +
             '.torch' + re.sub(r'[^0-9]', '', torch_base) + '.abi' + str(abi) +
             '.sm' + '_'.join(requested_sms))

ENV_INFO = {
    'created_utc': utc_now(), 'python_version': platform.python_version(),
    'python_tag': python_tag, 'python_executable': sys.executable,
    'os': platform.platform(), 'machine': platform.machine(),
    'torch_version': torch_full, 'torch_cuda': torch_cuda, 'torch_cxx11_abi': abi,
    'cuda_home': str(CUDA_HOME), 'nvcc_path': nvcc_path, 'nvcc_version': nvcc_version,
    'gpus': GPU_LIST, 'target_sms': requested_sms,
    'videomamba_commit': VIDEO_MAMBA_COMMIT,
    'mamba_ssm_source_version': '1.0.1', 'causal_conv1d_source_version': '1.0.0',
    'local_build_tag': local_tag, 'max_jobs': MAX_JOBS,
}
(OUTPUT_DIR / 'build_environment.json').write_text(json.dumps(ENV_INFO, indent=2) + '\n', encoding='utf-8')
print(json.dumps(ENV_INFO, indent=2))

## 3. Retrieve the exact VideoMamba sources

The pinned VideoMamba repository contains **both original CUDA extension sources** at `mamba/` and `causal-conv1d/`. Using these avoids accidentally installing an incompatible `mamba-ssm` package from PyPI.

You can optionally supply a Kaggle Input checkout via `CHECKOUT_SOURCE_FROM_INPUT` if network access is disabled. The source must be a **git checkout at the exact commit**. Otherwise Internet is required.

In [ ]:
if CHECKOUT_SOURCE_FROM_INPUT:
    supplied = Path(CHECKOUT_SOURCE_FROM_INPUT)
    if not supplied.is_dir():
        raise FileNotFoundError(f'Local repository not found: {supplied}')
    if REPO_DIR.exists():
        raise RuntimeError(f'Remove the old build checkout before copying the local source: {REPO_DIR}')
    shutil.copytree(supplied, REPO_DIR, symlinks=False)
else:
    if not (REPO_DIR / '.git').exists():
        run_logged(['git', 'clone', '--no-checkout', VIDEO_MAMBA_URL, REPO_DIR], '01_clone.log')
    run_logged(['git', '-C', REPO_DIR, 'fetch', '--depth', '1', 'origin', VIDEO_MAMBA_COMMIT], '02_fetch.log')
    run_logged(['git', '-C', REPO_DIR, 'checkout', '-f', VIDEO_MAMBA_COMMIT], '03_checkout.log')

actual_commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if actual_commit != VIDEO_MAMBA_COMMIT:
    raise RuntimeError(f'Wrong source commit {actual_commit} != {VIDEO_MAMBA_COMMIT}')
for part in ('mamba', 'causal-conv1d'):
    if not (REPO_DIR / part / 'setup.py').is_file():
        raise FileNotFoundError(f'Missing vendored source for {part}')
print('Source verification PASS:', actual_commit)

## 4. Minimal build dependencies

The historical `setup.py` scripts use the installed PyTorch headers and CUDA Toolkit. The build uses `--no-build-isolation --no-deps` so pip cannot replace the active PyTorch build. The build tools below are Python-only requirements; `nvcc` must already exist.

In [ ]:
run_logged([
    sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
    'packaging>=23', 'wheel>=0.42', 'setuptools>=70,<81', 'ninja', 'einops', 'psutil'
], '04_build_dependencies.log')

# Ensure the old source versions are those intended by VideoMamba. Fail on drift.
for sub, expect, module in [('mamba', '1.0.1', 'mamba_ssm'), ('causal-conv1d', '1.0.0', 'causal_conv1d')]:
    body = (REPO_DIR / sub / module / '__init__.py').read_text(encoding='utf-8')
    match = re.search(r'^__version__\s*=\s*[\"\']([^\"\']+)', body, re.MULTILINE)
    if not match or match.group(1) != expect:
        raise RuntimeError(f'{sub} version changed: expected {expect}, found {match.group(1) if match else "unknown"}')
print('Dependency source versions PASS: mamba-ssm==1.0.1, causal-conv1d==1.0.0')

## 5. Configure compilation for the detected GPU architecture

The original 2024 build scripts hardcode several architecture flags (e.g. `sm_70`, `sm_80`, `sm_90`). To avoid unnecessary compilation and include the **exact Kaggle GPU architecture**, this notebook modifies **only the compiler `-gencode` flags in a temporary checkout**. It preserves all CUDA kernels and BiMamba Python logic.

A source diff is saved as `source_setup_patches.diff` for reproducibility. If the upstream setup layout changes, the patch fails instead of silently editing a different section.

In [ ]:
patch_parts = []
patched_sha = {}
for sub in ('causal-conv1d', 'mamba'):
    p = REPO_DIR / sub / 'setup.py'
    original = p.read_text(encoding='utf-8')
    # Always start from the pinned git revision, even when re-running cells.
    original = subprocess.check_output(['git', '-C', str(REPO_DIR), 'show', f'{VIDEO_MAMBA_COMMIT}:{sub}/setup.py'], text=True)
    anchor = '    cc_flag.append("-gencode")'
    start = original.find(anchor)
    end = original.find('    # HACK:', start)
    if start < 0 or end <= start:
        raise RuntimeError(f'Unexpected architecture block layout in {sub}/setup.py; refusing an unsafe patch.')
    replacement = ''.join(
        f'    cc_flag.extend(["-gencode", "arch=compute_{sm},code=sm_{sm}"])\n'
        for sm in requested_sms
    ) + '\n'
    new_text = original[:start] + replacement + original[end:]
    p.write_text(new_text, encoding='utf-8')
    patched_sha[sub] = {'original_sha256': hashlib.sha256(original.encode()).hexdigest(),
                        'patched_sha256': sha256_file(p)}
    patch_parts.extend(difflib.unified_diff(
        original.splitlines(keepends=True), new_text.splitlines(keepends=True),
        fromfile=f'a/{sub}/setup.py', tofile=f'b/{sub}/setup.py'
    ))
patch_path = OUTPUT_DIR / 'source_setup_patches.diff'
patch_path.write_text(''.join(patch_parts), encoding='utf-8')
(OUTPUT_DIR / 'setup_source_hashes.json').write_text(json.dumps(patched_sha, indent=2) + '\n')
print('Patched for sm_' + ', sm_'.join(requested_sms))
print('Review compiler patch:', patch_path)
print(patch_path.read_text()[:4000])

## 6. Build `causal-conv1d` wheel

The `CAUSAL_CONV1D_FORCE_BUILD=TRUE` switch disables the old setup script's attempted download of a generic prebuilt wheel. Build output is streamed to the notebook **and** saved as `05_build_causal.log`.

In [ ]:
build_env = os.environ.copy()
build_env.update({
    'MAX_JOBS': str(MAX_JOBS),
    'TORCH_CUDA_ARCH_LIST': ';'.join(f'{sm[:-1]}.{sm[-1]}' for sm in requested_sms),
    'CAUSAL_CONV1D_FORCE_BUILD': 'TRUE',
    'MAMBA_FORCE_BUILD': 'TRUE',
    'CAUSAL_CONV1D_LOCAL_VERSION': local_tag,
    'MAMBA_LOCAL_VERSION': local_tag,
    'PIP_DISABLE_PIP_VERSION_CHECK': '1',
})

# Rebuilding replaces old wheels with the same package/version if present.
for old in OUTPUT_DIR.glob('causal_conv1d-*.whl'):
    old.unlink()
run_logged([
    sys.executable, '-m', 'pip', 'wheel', '--no-build-isolation', '--no-deps',
    '--no-cache-dir', '--wheel-dir', OUTPUT_DIR, str(REPO_DIR / 'causal-conv1d')
], '05_build_causal.log', env=build_env)
causal_wheels = sorted(OUTPUT_DIR.glob('causal_conv1d-*.whl'))
if len(causal_wheels) != 1:
    raise RuntimeError(f'Expected one causal-conv1d wheel, found {len(causal_wheels)}: {causal_wheels}')
print('CAUSAL BUILD PASS:', causal_wheels[0].name, '| bytes:', causal_wheels[0].stat().st_size)

## 7. Build original BiMamba (`mamba-ssm`) wheel

`MAMBA_FORCE_BUILD=TRUE` forces compilation from **VideoMamba's fork**. Do not substitute a newer PyPI Mamba wheel: BiMamba compatibility is tested below.

In [ ]:
for old in OUTPUT_DIR.glob('mamba_ssm-*.whl'):
    old.unlink()
run_logged([
    sys.executable, '-m', 'pip', 'wheel', '--no-build-isolation', '--no-deps',
    '--no-cache-dir', '--wheel-dir', OUTPUT_DIR, str(REPO_DIR / 'mamba')
], '06_build_mamba.log', env=build_env)
mamba_wheels = sorted(OUTPUT_DIR.glob('mamba_ssm-*.whl'))
if len(mamba_wheels) != 1:
    raise RuntimeError(f'Expected one mamba-ssm wheel, found {len(mamba_wheels)}: {mamba_wheels}')
print('MAMBA BUILD PASS:', mamba_wheels[0].name, '| bytes:', mamba_wheels[0].stat().st_size)

## 8. Validate wheel filenames, tags, and binary contents

Wheel tags cover **Python ABI and operating-system architecture**. PyTorch/CUDA/GPU dependencies are recorded separately in the manifest because standard wheel tags cannot reliably encode them. We also verify each archive contains the correct compiled `.so` extension.

In [ ]:
from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename

def inspect_wheel(p, expected_name, expected_so):
    dist, version, build_tag, tags = parse_wheel_filename(p.name)
    if str(dist).replace('-', '_').lower() != expected_name:
        raise RuntimeError(f'Wrong distribution in wheel {p.name}')
    if not (set(tags) & set(sys_tags())):
        raise RuntimeError(f'Wheel does not match this interpreter / platform: {p.name}')
    if not zipfile.is_zipfile(p):
        raise RuntimeError(f'Invalid .whl ZIP archive: {p.name}')
    with zipfile.ZipFile(p) as z:
        bad = z.testzip()
        if bad: raise RuntimeError(f'Corrupt wheel {p.name}, bad member {bad}')
        objects = [x for x in z.namelist() if expected_so in Path(x).name and x.endswith('.so')]
        if not objects:
            raise RuntimeError(f'Missing compiled CUDA library {expected_so} in {p.name}')
    return {'file': p.name, 'version': str(version), 'bytes': p.stat().st_size,
            'sha256': sha256_file(p), 'binary_extensions': objects}

WHEELS = {
    'causal_conv1d': inspect_wheel(causal_wheels[0], 'causal_conv1d', 'causal_conv1d_cuda'),
    'mamba_ssm': inspect_wheel(mamba_wheels[0], 'mamba_ssm', 'selective_scan_cuda'),
}
print(json.dumps(WHEELS, indent=2))
print('BUILD PASS: both archived CUDA wheels contain native extensions.')

## 9. Install compiled wheels and run CUDA smoke tests

This is a **separate Python subprocess**, so it avoids stale kernel imports. The smoke test checks:

- importing `causal_conv1d_cuda` and `selective_scan_cuda`;
- that `Mamba.__init__` includes **`bimamba`**;
- short CUDA forward/backward passes for causal-conv1d and BiMamba;
- finite tensors and non-null gradients.

`transformers==4.36.1` is the historical dependency used with this original BiMamba implementation. It may be unavailable or otherwise incompatible on a newly upgraded Python version. Such failures mean **the full training environment is not yet ready**, even if both CUDA wheel builds succeeded.

In [ ]:
if not INSTALL_BUILT_WHEELS and RUN_GPU_SMOKE_TEST:
    raise ValueError('RUN_GPU_SMOKE_TEST requires INSTALL_BUILT_WHEELS=True')

if INSTALL_BUILT_WHEELS:
    # Avoid silently pulling in / replacing torch while installing binary extensions.
    for i, whl in enumerate([causal_wheels[0], mamba_wheels[0]]):
        run_logged([sys.executable, '-m', 'pip', 'install', '--no-deps', '--force-reinstall',
                    '--no-cache-dir', str(whl)], f'07_install_{i+1}.log')

if INSTALL_LEGACY_IMPORT_DEPS and INSTALL_BUILT_WHEELS:
    run_logged([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                'einops==0.7.0', 'transformers==4.36.1'], '08_import_dependencies.log')

smoke_script = r"""
import sys, json, inspect
import torch
import causal_conv1d_cuda, selective_scan_cuda
from causal_conv1d import causal_conv1d_fn
from mamba_ssm.modules.mamba_simple import Mamba
assert 'bimamba' in inspect.signature(Mamba.__init__).parameters, 'This Mamba package lacks VideoMamba BiMamba support.'
assert torch.cuda.is_available()
x = torch.randn(1, 8, 24, device='cuda', dtype=torch.float32, requires_grad=True)
w = torch.randn(8, 4, device='cuda', dtype=torch.float32, requires_grad=True)
y = causal_conv1d_fn(x, w)
assert tuple(y.shape) == (1, 8, 24) and torch.isfinite(y).all().item()
y.float().square().mean().backward()
assert x.grad is not None and w.grad is not None
model = Mamba(d_model=32, d_state=8, d_conv=4, expand=2, bimamba=True).cuda().float().train()
features = torch.randn(1, 16, 32, device='cuda', dtype=torch.float32, requires_grad=True)
out = model(features)
assert tuple(out.shape) == (1, 16, 32) and torch.isfinite(out).all().item(), out.shape
out.float().square().mean().backward()
assert features.grad is not None and torch.isfinite(features.grad).all().item()
torch.cuda.synchronize()
print(json.dumps({'status':'RUNTIME PASS', 'python':sys.version.split()[0], 'torch':torch.__version__,
                  'torch_cuda':torch.version.cuda, 'gpu':torch.cuda.get_device_name(0),
                  'bimamba':True, 'causal_forward_backward':True, 'mamba_forward_backward':True}, indent=2))
"""
(OUTPUT_DIR / 'cuda_smoke_test.py').write_text(smoke_script, encoding='utf-8')
RUNTIME_VALIDATED = False
if RUN_GPU_SMOKE_TEST:
    rc = run_logged([sys.executable, str(OUTPUT_DIR / 'cuda_smoke_test.py')], '09_cuda_smoke_test.log', check=False)
    RUNTIME_VALIDATED = rc == 0
    if not RUNTIME_VALIDATED:
        print('RUNTIME FAIL. Both wheels may have built, but CUDA/BiMamba verification failed. Do not reuse them as validated wheels.')
        print('See:', OUTPUT_DIR / '09_cuda_smoke_test.log')
    else:
        print('RUNTIME PASS: Both native modules and BiMamba backward work in this runtime.')
else:
    print('RUNTIME NOT TESTED: wheels are not yet validated for training.')

## 10. Create the reproducibility manifest and future-session installer

The installer checks **exact Python version family/ABI, PyTorch version, PyTorch CUDA version, C++11 ABI, supported GPU architecture and wheel SHA-256** before installing the two wheels. It will **refuse an upgraded Kaggle runtime** rather than attempt a potentially unsafe load. For a changed runtime, rerun this *builder notebook* and publish a new wheel dataset version.

In [ ]:
import textwrap

manifest = {
    'schema_version': 1,
    'source': {'repository': VIDEO_MAMBA_URL, 'commit': VIDEO_MAMBA_COMMIT,
               'vendored_sources': {'causal_conv1d': 'causal-conv1d/', 'mamba_ssm': 'mamba/'},
               'setup_patches_sha256': sha256_file(patch_path)},
    'environment': ENV_INFO,
    'wheels': WHEELS,
    'verification': {
        'wheel_archives_passed': True,
        'installed_to_builder_runtime': INSTALL_BUILT_WHEELS,
        'cuda_smoke_test_requested': RUN_GPU_SMOKE_TEST,
        'cuda_smoke_test_passed': RUNTIME_VALIDATED,
    },
}
OUTPUT_MANIFEST.write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8')

installer_script = r"""
#!/usr/bin/env python3
# Install VideoMamba native-wheel pair only into a matching Kaggle runtime.
# Usage: python verify_install.py /kaggle/input/.../wheel_manifest.json
import sys, json, hashlib, subprocess
from pathlib import Path
import torch
from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename

def digest(p):
    h = hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024), b''): h.update(chunk)
    return h.hexdigest()

manifest_path = Path(sys.argv[1]).resolve() if len(sys.argv)>1 else Path(__file__).with_name('wheel_manifest.json')
if not manifest_path.is_file(): raise FileNotFoundError(manifest_path)
m = json.loads(manifest_path.read_text(encoding='utf-8'))
if not m['verification'].get('cuda_smoke_test_passed'):
    raise RuntimeError('Wheel pair was not CUDA-validated by its builder. Refusing installation.')
e = m['environment']
expected_python_tag = e['python_tag']
actual_python_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
if actual_python_tag != expected_python_tag:
    raise RuntimeError(f'Python ABI mismatch: expected {expected_python_tag}, got {actual_python_tag}')
if torch.__version__ != e['torch_version']:
    raise RuntimeError(f'PyTorch mismatch: expected {e["torch_version"]}, got {torch.__version__}')
if str(torch.version.cuda) != str(e['torch_cuda']):
    raise RuntimeError(f'PyTorch CUDA mismatch: expected {e["torch_cuda"]}, got {torch.version.cuda}')
if int(bool(torch._C._GLIBCXX_USE_CXX11_ABI)) != int(e['torch_cxx11_abi']):
    raise RuntimeError('PyTorch C++11 ABI mismatch')
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not active')
actual_sms = {''.join(map(str, torch.cuda.get_device_capability(i))) for i in range(torch.cuda.device_count())}
if not actual_sms.issubset(set(e['target_sms'])):
    raise RuntimeError(f'Unsupported current GPU SM(s): {sorted(actual_sms)} vs built {e["target_sms"]}')
for name in ('causal_conv1d', 'mamba_ssm'):
    wheel = m['wheels'][name]
    file = manifest_path.parent / wheel['file']
    if not file.is_file() or digest(file) != wheel['sha256']:
        raise RuntimeError(f'Missing or corrupt wheel {file}')
    _, _, _, tags = parse_wheel_filename(file.name)
    if not (set(tags) & set(sys_tags())):
        raise RuntimeError(f'Wheel tag incompatible: {file.name}')
    subprocess.run([sys.executable,'-m','pip','install','--no-deps','--force-reinstall',str(file)],check=True)
print('INSTALL PASS: matching Python / Torch / CUDA / ABI / GPU SM and wheel hashes.')
print('Install the historical import dependencies (transformers==4.36.1, einops==0.7.0) and run cuda_smoke_test.py to verify this new runtime.')
"""
(OUTPUT_DIR / 'verify_install.py').write_text(installer_script, encoding='utf-8')

readme = f"""# VideoMamba CUDA wheels — pinned BiMamba fork

- Source commit: `{VIDEO_MAMBA_COMMIT}`
- CUDA extensions: `causal-conv1d 1.0.0`, `mamba-ssm 1.0.1` (vendored VideoMamba fork)
- Python: `{ENV_INFO['python_version']}` ({python_tag})
- Torch: `{torch_full}`; PyTorch CUDA `{torch_cuda}`; nvcc `{nvcc_version}`
- C++11 ABI: `{abi}`; GPU targets: `{', '.join('sm_'+x for x in requested_sms)}`
- Native wheel archive validation: PASS
- CUDA forward/backward runtime smoke test: {'PASS' if RUNTIME_VALIDATED else 'NOT PASSED'}

## Install in a matching Kaggle notebook

Attach the published wheel dataset as Kaggle Input and run:

```python
!python /kaggle/input/<OWNER_OR_DATASET_PATH>/verify_install.py /kaggle/input/<OWNER_OR_DATASET_PATH>/wheel_manifest.json
```

Replace the path with the **actual** Add Input mount path. The installer intentionally refuses a mismatched environment. The EgoEvGesture Train notebook can also discover these two wheels by scanning `/kaggle/input`.

**Install `einops==0.7.0` and `transformers==4.36.1` if not already provided by the Train notebook.** A successful wheel build does not resolve independent Python package compatibility problems.

## Publishing

Save your Kaggle notebook output, then create a private Kaggle Dataset from **this output folder** (`/kaggle/working/videomamba_cuda_wheels`). Include both wheels, manifest, verify_install.py and logs. Do not publish a build unless `cuda_smoke_test_passed=true`.

## Important

These wheels are native code and are **not universal**. Build again when Kaggle changes Python ABI, PyTorch binary, CUDA stack, C++ ABI, or GPU targets. The build may fail under future Python/PyTorch/CUDA releases because this is pinned 2024-era source; such source compatibility is outside wheel metadata.
"""
(OUTPUT_DIR / 'README.md').write_text(readme, encoding='utf-8')

print('Manifest saved:', OUTPUT_MANIFEST)
print('Validated:', RUNTIME_VALIDATED)
print('Output contents:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {path.name:<55} {path.stat().st_size:>12,} bytes')

## 11. Final publication gate

- **BUILD PASS**: both wheels exist with valid native extension files.
- **RUNTIME PASS**: both compiled modules support actual GPU forward/backward execution and **BiMamba**.
- **NOT VERIFIED / FAIL**: do **not** treat the wheels as safe for training; review the saved build/smoke logs.

**Reuse workflow:**

1. Run this notebook on Kaggle with Internet + GPU + `nvcc` available.
2. Confirm **`RUNTIME PASS`** in the cell below.
3. Save the notebook output and publish `/kaggle/working/videomamba_cuda_wheels/` to a private Kaggle Dataset, e.g. `videomamba-cuda-wheels-python312-torchXXX-cuXXX`.
4. In the EgoEvGesture Train notebook, **Add Input** this wheel dataset alongside the original event dataset and validated split dataset.
5. If Kaggle later upgrades Python, PyTorch, or CUDA, run this builder again; **never assume an old `.whl` is portable**.

This notebook does not upload to Kaggle automatically, does not modify the original event dataset, and does not run model training/testing.

In [ ]:
result = json.loads(OUTPUT_MANIFEST.read_text(encoding='utf-8'))
assert result['verification']['wheel_archives_passed'], 'Wheel archive verification failed.'
print('BUILD PASS: both CUDA wheels were created from the pinned VideoMamba sources.')
if result['verification']['cuda_smoke_test_passed']:
    print('RUNTIME PASS: verified CUDA forward/backward and VideoMamba BiMamba compatibility.')
    print('READY TO PUBLISH: create a private Kaggle Dataset from', OUTPUT_DIR)
else:
    raise RuntimeError('RUNTIME NOT VERIFIED: do not publish these as training-ready wheels. Review the log files and repair the environment.')